Utilizando esse conjunto de dados: https://www.kaggle.com/datasets/disham993/9000-movies-dataset

Responda as seguintes perguntas:

In [1]:
# Manipulação e visualização de dados
import pandas as pd
import numpy as np
import os
# Bibliotecas para aprendizado de máquina
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# MLflow para rastreamento de experimentos
import mlflow

# Supressão de avisos
import warnings
warnings.filterwarnings("ignore")

#### Puxar o Dataset pela API

Precisei fazer uma pré verificação para conseguir puxar o arquivo csv do kaggle, pois estava tomando erro de arquivo mal formatado.

In [8]:
# Baixar e descompactar o dataset na pasta ./data
os.makedirs("data2", exist_ok=True)
!kaggle datasets download -d disham993/9000-movies-dataset -p ./data2 --unzip

# Listar todos os arquivos baixados
print("\n Arquivos na pasta data2/:")
for f in os.listdir("data2"):
    print(" -", f)

# Procurar o primeiro arquivo CSV encontrado
csv_files = [f for f in os.listdir("data2") if f.endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado na pasta 'data2/'")

# Ver primeiras linhas do arquivo cru (para ver o formato real)
csv_path = os.path.join("data2", csv_files[0])
print(f"\n Verificando arquivo: {csv_files[0]}\n")
with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(5):
        print(f.readline().strip())

# Carregar o CSV com tolerância extra
try:
    df = pd.read_csv(
        csv_path,
        engine="python",      # mais tolerante
        on_bad_lines="skip",  # ignora linhas ruins
        encoding="utf-8",     # tenta UTF-8
        sep=","             # assume separador vírgula
    )
except Exception as e:
    print("Não conseguiu ler o arquivo")

print(f"\n Arquivo carregado: {csv_files[0]}")

Dataset URL: https://www.kaggle.com/datasets/disham993/9000-movies-dataset
License(s): CC0-1.0


 Arquivos na pasta data2/:
 - mymoviedb.csv

 Verificando arquivo: mymoviedb.csv

Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
2021-12-15,Spider-Man: No Way Home,"Peter Parker is unmasked and no longer able to separate his normal life from the high-stakes of being a super-hero. When he asks for help from Doctor Strange the stakes become even more dangerous, forcing him to discover what it truly means to be Spider-Man.",5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg
2022-03-01,The Batman,"In his second year of fighting crime, Batman uncovers corruption in Gotham City that connects to his own family while facing a serial killer known as the Riddler.",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R36Fpooo50r9T25onhq.jp


  0%|          | 0.00/1.70M [00:00<?, ?B/s]
100%|██████████| 1.70M/1.70M [00:00<00:00, 530MB/s]


In [9]:
# Mostrar as primeiras linhas
df.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


#### Qual tamanho do DataSet?

88533

In [10]:
print(f"Tamanho do DataSet: {df.size}")

Tamanho do DataSet: 88533


#### Quantas linhas e colunas?

Linhas: 9837

Colunas: 9

In [11]:
print(f"Quantidade de linhas e colunas: {df.shape}")

Quantidade de linhas e colunas: (9837, 9)


#### Qual o tipo de variável de cada coluna?

Release_Date:          object

Title:                 object

Overview:              object

Popularity:           float64

Vote_Count:            object

Vote_Average:          object

Original_Language:     object

Genre:                 object

Poster_Url:            object

In [12]:
df.dtypes

Release_Date          object
Title                 object
Overview              object
Popularity           float64
Vote_Count            object
Vote_Average          object
Original_Language     object
Genre                 object
Poster_Url            object
dtype: object

#### Qual o filme com maior número de votações?

É o filme: Inception com 31077 votos

In [13]:
# Precisei realizar a conversão da coluna Vote_count para int, para poder fazer o max(), porém ele tinha alguns dados nulos que transformei para 0 para facilitar a analise
df['Vote_Count'] = pd.to_numeric(df['Vote_Count'], errors='coerce').fillna(0).astype(int)

In [14]:
top_movie = df.loc[df['Vote_Count'].idxmax(), ['Title', 'Vote_Count']]
print(top_movie)

Title         Inception
Vote_Count        31077
Name: 380, dtype: object


#### Qual filme teve a maior nota (critério de desempate é o filme com mais votos)

Foi o filme: Kung Fu Master Huo Yuanjia

In [15]:
# Ajustando a coluna de nota para poder fazer uma analise melhor
df['Vote_Average'] = pd.to_numeric(df['Vote_Average'], errors='coerce').fillna(0).astype(float)

In [16]:
# Verificando os top 5 filmes por ordem de maior nota
top5 = df.sort_values(by=['Vote_Average', 'Vote_Count'], ascending=[False, False]).head(5)
top5[['Title', 'Vote_Average', 'Vote_Count']]

,Title,Vote_Average,Vote_Count
9401,Kung Fu Master Huo Yuanjia,10.0,1
7349,Franco Escamilla: Por La Anécdota,9.2,92
2335,Impossible Things,9.1,82
667,Demon Slayer: Kimetsu no Yaiba Sibling's Bond,9.1,27
2401,The Three Deaths of Marisela Escobedo,9.0,183


In [17]:
# Verificando de outra forma o filme com maior nota
top_average = df.loc[df['Vote_Average'].idxmax(), ['Title', 'Vote_Average']]
print(top_average)

Title           Kung Fu Master Huo Yuanjia
Vote_Average                          10.0
Name: 9401, dtype: object


#### Existem valores nulos? Se sim, qual tratamento irá realizar? (Se não temos nome de algum filme, melhor nem considerar)

Inicialmente existiam 6 colunas com valores Nulos, como uma delas é a coluna Title realizei a remoção dos valores nulos baseados nela.

Após remover os nulos de Title, ainda sobrou 4 colunas com alguns valores nulos, então realizei tratamentos de preenchimento:

Para a coluna Genre optei por preencher os nulos como "Unknown"

Para a coluna Poster_Url optei por preencher os nulos como "No poster available"

Para a coluna Popularity optei por usar o método de preenchimento usando a mediana dos resultados para preenchimento.

Para a coluna Original_Language optei por usar o método de preenchimento usando a moda dos resultados para preenchimento.

In [18]:
# Verificar valores ausentes
display(df.isnull().sum().sort_values(ascending=False))


Genre                11
Poster_Url           11
Popularity           10
Original_Language    10
Title                 9
Overview              9
Release_Date          0
Vote_Count            0
Vote_Average          0
dtype: int64

In [19]:
# Descartando os titulos nulos, pois não teriamos como adivinhar qual filme é
df = df.dropna(subset=['Title'])

In [20]:
# Verificando valores ausentes após tropar titulos nulos
display(df.isnull().sum().sort_values(ascending=False))


Genre                2
Poster_Url           2
Popularity           1
Original_Language    1
Release_Date         0
Vote_Count           0
Overview             0
Title                0
Vote_Average         0
dtype: int64

In [21]:
# Categóricas simples
df['Genre'] = df['Genre'].fillna('Unknown')
df['Poster_Url'] = df['Poster_Url'].fillna('No poster available')

# Numérica - Popularity
df['Popularity'] = df['Popularity'].fillna(df['Popularity'].median())

# Categórica - Original_Language
df['Original_Language'] = df['Original_Language'].fillna(df['Original_Language'].mode()[0])


In [22]:
# Verificando valores ausentes após tratamento das colunas
display(df.isnull().sum().sort_values(ascending=False))

Release_Date         0
Title                0
Overview             0
Popularity           0
Vote_Count           0
Vote_Average         0
Original_Language    0
Genre                0
Poster_Url           0
dtype: int64

Transforme as variaveis categóricas de linguagem e genero em númericas (utilize dummy)

In [23]:
# Criar dummies para as colunas categóricas
df_dummies = pd.get_dummies(df, columns=['Original_Language', 'Genre'], drop_first=True, dtype=int)
# Verificar o resultado
df_dummies.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Poster_Url,Original_Language_bn,Original_Language_ca,Original_Language_cn,...,"Genre_Western, Drama","Genre_Western, Drama, Action, Adventure","Genre_Western, Drama, Adventure","Genre_Western, Drama, Crime","Genre_Western, Drama, History","Genre_Western, Drama, Mystery","Genre_Western, History","Genre_Western, Horror","Genre_Western, Mystery, Thriller, Drama","Genre_Western, Thriller"
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,https://image.tmdb.org/t/p/original/74xTEgt7R3...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Normalize as variaveis numéricas

In [24]:
dados_scaled_minmax = df_dummies.copy()
dados_scaled_standard = df_dummies.copy()

num_cols = df_dummies.select_dtypes(include=[np.number]).columns

minmax = MinMaxScaler()
dados_scaled_minmax[num_cols] = minmax.fit_transform(dados_scaled_minmax[num_cols])

standard = StandardScaler()
dados_scaled_standard[num_cols] = standard.fit_transform(dados_scaled_standard[num_cols])

print('Visualização após Normalização (Min-Max):')
display(dados_scaled_minmax[num_cols].describe().T.head())

print('Visualização após Padronização (Standard):')
display(dados_scaled_standard[num_cols].describe().T.head())

Visualização após Normalização (Min-Max):


,count,mean,std,min,25%,50%,75%,max
Popularity,9828.0,0.006543,0.021444,0.0,0.001778,0.002776,0.005529,1.0
Vote_Count,9828.0,0.044813,0.084021,0.0,0.004698,0.014287,0.044277,1.0
Vote_Average,9828.0,0.643816,0.113341,0.0,0.590000,0.650000,0.710000,1.0
Original_Language_bn,9828.0,0.000102,0.010087,0.0,0.000000,0.000000,0.000000,1.0
Original_Language_ca,9828.0,0.000102,0.010087,0.0,0.000000,0.000000,0.000000,1.0


Visualização após Padronização (Standard):


,count,mean,std,min,25%,50%,75%,max
Popularity,9828.0,4.627059e-17,1.000051,-0.305140,-0.222213,-0.175703,-0.047297,46.329945
Vote_Count,9828.0,2.313529e-17,1.000051,-0.533386,-0.477468,-0.363335,-0.006381,11.369038
Vote_Average,9828.0,-1.995419e-16,1.000051,-5.680629,-0.474836,0.054567,0.583970,3.142750
Original_Language_bn,9828.0,1.084467e-18,1.000051,-0.010088,-0.010088,-0.010088,-0.010088,99.131226
Original_Language_ca,9828.0,-9.037224e-18,1.000051,-0.010088,-0.010088,-0.010088,-0.010088,99.131226


Armazene esses valores como um artefato dentro do MLFlow

In [25]:
# Salvar
processed_data_path = "dados_filmes.csv"
df.to_csv(processed_data_path, index=False)
print("Dataset processado salvo localmente.")

Dataset processado salvo localmente.


In [26]:
# Registrar o dataset processado como artefato no MLflow
mlflow.start_run()  # Iniciar um novo experimento
mlflow.log_artifact(processed_data_path)  # Registrar o arquivo como artefato
mlflow.end_run()  # Encerrar o experimento

print("Features armazenadas e versionadas com sucesso no MLflow!")

Features armazenadas e versionadas com sucesso no MLflow!


#### Quais insights é possivel obter desses dados?

Podemos ver que nem todo filme popular é o filme com mais votos ou melhor classificado devido a diferença de avaliação para filmes menos popules/votados.